# ETF_2_Code — Playbook PDF parser (page 1 + page 2..N)

Extends `ETF_1_Code_v3_UBugfix2.ipynb` to also parse page 2 of the Playbook PDF (the
"Featured Trades" page — see `docs/Design-Plan.md`). Page 2 mixes:

- a trade table (`"Featured Trade & Maintenance Suggestions"` / `"Featured Investor &
  Maintenance Suggestions"`) which feeds the same row-parsing logic already used for
  pages 3+ (Bond ETF, International ETF, ...), extended here to also handle multi-leg
  combo trades (e.g. a Bull Call Spread + Naked Put sharing one net Entry/Target/Stop)
  and a `quantity` column for `BTO`/`STO` contract counts.
- trailing free-text market commentary, which is split off and saved locally
  (Feature A / llmwiki ingestion is deferred until llmwiki is deployed — see
  `docs/TODOS.md` — but the retry/skip-designed upload function is included so wiring
  it in later is a one-line change).

The parsing logic in the main loop below is otherwise unchanged from
`ETF_1_Code_v3_UBugfix2.ipynb` cell 9 — only the additions called out in comments are
new.


In [ ]:
#Importing required libraries
import traceback
import pandas as pd
import re
import pdfplumber
from collections import namedtuple
from calendar import month_name
import fitz
import dataUtil as DU
import logging
import requests
import os


## Locate & read the source PDF

`Playbook-2026-09-07.pdf` (checked into the repo root) is the confirmed sample/reference
input for this notebook (see `docs/Design-Plan.md`). For a production run, swap this for
the Google-Drive-mounted path the same way `ETF_1_Code_v3_UBugfix2.ipynb` does (candidate
paths commented out below) and update `InputDate` to match.


In [ ]:
#Import PDF file from path
InputDate = '2026-09-07'
currentLCR = f'912 Playbook {InputDate}.pdf'
# Production path (Google-Drive mounted in WSL) — same convention as ETF_1_Code_v3_UBugfix2.ipynb:
#   Ubuntu/WSL file path location
file=f"/mnt/i/My Drive/ReadProjects/Neural Matrix Investment/LCR/Playbooks/{currentLCR}"
# Sample/reference PDF checked into the repo root, used for building/testing this notebook:
# file = "Playbook-2026-09-07.pdf"
print('Processing LCR:', file)

In [ ]:
from pathlib import Path

file_path = Path(file)
if file_path.is_file():
    print(f"File exists: {file_path}")
else:
    print(f"File does not exist: {file_path}")

In [ ]:
# Page 1 (Disclosure Statement) is boilerplate and is still skipped. Page 2 (the
# "Featured" page) is now included, but it mixes the trade table with trailing market
# commentary, so it needs to be split before either half is used:
#   - table half   -> appended to text_full, feeds the same parser as pages 3+
#   - commentary half -> captured separately for Feature A (llmwiki ingestion)
commentary_text = ""
with fitz.open(file) as doc:
    text_full = ""
    c = 0
    for page in doc:
        c += 1
        if c < 2: continue   # skip page 1 (Disclosure Statement) only
        text = page.get_text()
        if c == 2:
            # Split on the "Copyright ..." line: everything before it is the Featured
            # trade table (TOC preamble + trades), everything after it is the free-text
            # market commentary ("Market Expectations for ..."). Confirmed against
            # Playbook-2026-09-07.pdf page 2 — see docs/Design-Plan.md.
            split_marker = "Copyright"
            idx = text.find(split_marker)
            if idx > -1:
                table_part = text[:idx]
                commentary_text = text[idx:].split("\n", 1)[-1].strip()
            else:
                table_part = text
                logging.warning(
                    "Expected 'Copyright' marker not found on page 2; treating the "
                    "whole page as trade table text, no commentary captured."
                )
            text = table_part
        print(f"==> page: {c}", text)
        text_full += text
print(text_full)


In [ ]:
#Basic columns to be read from PDF
col = ["Type", "Symbol", "Trend", "Status", "Expiration", "PnC", "L_Strike", "H_Strike",
       "Entry", "Target", "Stop", "quantity"]
data_f = []
for x in col:
    data_f.append(1 if x == "quantity" else '')
df = pd.DataFrame([data_f], columns=col)
df.head()


## Find where the real trade-table text starts

Generalized from `"Bond ETF Trade & Maintenance Suggestions"` (the original, page-3-only
marker) to `"Trade & Maintenance Suggestions"` so the first match is now
`"Featured Trade & Maintenance Suggestions"` on page 2 — this is the confirmed split
point from `docs/Design-Plan.md` and lets the existing skip-until-`num` guard in the main
loop keep working unchanged, just starting one section earlier.


In [ ]:
iterates = iter(text_full.split('\n'))
num = 0
for line in iterates:
    num += 1
    if line.find("Trade & Maintenance Suggestions") >= 0:
        break
print(line)
print(num)


## Main parsing loop

Same structure as `ETF_1_Code_v3_UBugfix2.ipynb` cell 9, with three additions (each
called out in a comment at its point of use):

1. **`quantity`** — `BTO`/`STO` plus an optional trailing contract count, signed by
   direction (`STO 2` -> `-2`, `BTO` alone -> `1`). Strikes are now parsed from the text
   *after* this action/count prefix so a leading count is never mistaken for a strike
   (e.g. `"STO 2 March 47.5 Naked Puts"` previously would have read `2` and `47.5` as two
   strikes).
2. **Multi-leg combo entry** — a new `"Entry of a net <value> Credit|Debit"` branch
   (vs. the existing `"Entry @ <price>, ..."` branch, left untouched). `Entry` stays
   numeric: the value is negated for `"Credit"`, kept positive for `"Debit"`.
3. **Combo broadcast** — once a combo is detected, `Entry`/`Target`/`Stop` are copied to
   every leg-row parsed since the current Trend group started (`group_start_row`), not
   just the row that happened to contain the entry/stop line. This only changes behavior
   for the new combo case — the existing single-leg `"Entry @ "` path (pages 3-8) is
   unchanged, so `Target_Sign`/`Stop_Sign`/`Stop`-value backfill for those pages still
   comes from the existing cells below.


In [ ]:
iterates = iter(text_full.split('\n'))
Type_name = None
Stock_name_tmp = None
Status = None
Trend = None
line_num = -1
row_number = -1
group_start_row = 0     # first row index of the current Trend group (for combo broadcast)
is_combo = False        # set once an "Entry of a net ... Credit/Debit" line is seen
checkk = 0
for line in iterates:
    print(line, checkk, num)
    checkk += 1
    if checkk+1 > num:
        try:
            if line.find("Trade & Maintenance Suggestions") > 0:
                ch1 = line.find("Trade & Maintenance Suggestions")
                Type_name = line[:ch1].strip()
                print(f'Type_name = {Type_name}')
            else:
                if line.strip() == '' or line.find("Aggressive Traders:") > -1:
                    continue
                if line.find("Trend") == -1 and re.search(r"\[\w*\]", line):
                    Stock_name_tmp = (re.findall(r"\[\w*\]", line)[0][1:-1])
                    line_num = 0
                    print(f'Stock_name is {Stock_name_tmp}')
                else:
                    if line_num == 0:
                        Trend = re.findall(r"Investor|Trader", line)[0]
                        Status = re.findall(r"\[\S*\]", line)[0][1:-1]
                        line_num = 1
                        group_start_row = row_number + 1
                        is_combo = False
                    else:
                        if (line.find("Aggressive Traders:") > -1):
                            pass
                        elif (line.find("Trend") > 0):
                            Trend = re.findall(r"Investor|Trader", line)[0]
                            Status = re.findall(r"\[\S*\]", line)[0][1:-1]
                            group_start_row = row_number + 1
                            is_combo = False
                        elif ((line.find("STO") > -1) or (line.find("BTO") > -1)):
                            row_number += 1
                            df.loc[len(df.index)] = data_f
                            df['Type'][row_number] = Type_name
                            df['Symbol'][row_number] = Stock_name_tmp
                            df['Trend'][row_number] = Trend
                            df['Status'][row_number] = Status
                            decission = re.findall(r"Put|Call", line)[0][0]
                            df['PnC'][row_number] = decission

                            # --- new: BTO=Long/Buy, STO=Short/Sell; trailing number is the
                            # contract count, signed by direction (STO 2 -> -2) ---
                            action_match = re.search(r"\b(BTO|STO)\b\s*(\d+)?", line)
                            action = action_match.group(1)
                            qty = int(action_match.group(2)) if action_match.group(2) else 1
                            df['quantity'][row_number] = qty if action == "BTO" else -qty

                            print('** Find Exp Date **', line)
                            exp_pattern = r'\((\d{1,2}/\d{1,2}/\d{2,4})\s*exp\)'
                            matches = re.findall(exp_pattern, line)
                            print(" new found exp date: ", matches)
                            exp_date = matches[0]
                            df['Expiration'][row_number] = exp_date

                            # --- changed: strikes are parsed from the text *after* the
                            # action/count prefix, so a leading contract count is never
                            # mistaken for a strike (e.g. "STO 2 March 47.5 Naked Puts") ---
                            remainder = line[action_match.end():]
                            price = remainder.split('(')[0]
                            matches = re.finditer(r"\d+(\.)?\d+", price)
                            print(f'matches : {matches}')
                            tmp_price = []
                            for m in matches:
                                tmp_price.append(m[0])
                            l_price = None
                            h_price = tmp_price[0]
                            if len(tmp_price) == 2:
                                l_price = tmp_price[0]
                                h_price = tmp_price[1]
                            df["L_Strike"][row_number] = l_price
                            df["H_Strike"][row_number] = h_price
                        elif (line.find("Entry of a net") > -1):
                            # --- new: multi-leg combo (e.g. Bull Call Spread + Naked Put
                            # sharing one net Entry/Target/Stop). Entry stays numeric:
                            # negate for "Credit", keep positive for "Debit". Broadcast to
                            # every leg-row parsed since this Trend group started. ---
                            is_combo = True
                            net_match = re.search(
                                r"Entry of a net\s+([\d.]+)\s+(Credit|Debit)", line, re.IGNORECASE)
                            net_val = float(net_match.group(1))
                            entry_val = -net_val if net_match.group(2).lower() == "credit" else net_val
                            target_val = None
                            if ',' in line:
                                target_part = line.split(',')[1]
                                trgt = [z[0] for z in re.finditer(r"\d+(\.)?\d+", target_part)]
                                if trgt:
                                    target_val = trgt[0]
                            for r in range(group_start_row, row_number + 1):
                                df["Entry"][r] = entry_val
                                if target_val is not None:
                                    df["Target"][r] = target_val
                        elif (line.find("Entry @ ") > -1):
                            entry = line.split(',')[0]
                            mtch_entry = re.finditer(r"\d+(\.)?\d+", entry)
                            ent = []
                            for n in mtch_entry:
                                ent.append(n[0])
                            df["Entry"][row_number] = ent[0]
                            target = line.split(',')[1]
                            mtch_trgt = re.finditer(r"\d+(\.)?\d+", target)
                            trgt = []
                            for z in mtch_trgt:
                                trgt.append(z[0])
                            df["Target"][row_number] = trgt[0]
                        elif (line.find("Stop") > -1):
                            if line.count("@") > 1:
                                stop = line.split('@', 2)[2]
                            else:
                                stop = line.split('@')[1]
                            mtch_stop = re.finditer(r"\d+(\.)?\d+", stop)
                            stp = []
                            for q in mtch_stop:
                                stp.append(q[0])
                            # --- changed: for a combo, broadcast Stop to every leg-row of
                            # the group instead of just the current (last) row ---
                            if is_combo:
                                for r in range(group_start_row, row_number + 1):
                                    df["Stop"][r] = stp[0]
                            else:
                                df["Stop"][row_number] = stp[0]
                        else:
                            pass
        except Exception as e:
            print(f'Exception on line:{line} => {e}')
print(df)


In [ ]:
df.head()


In [ ]:
df = df[:-1]
df["Low"] = ''
df["High"] = ''
df["Low-High"] = ''

#Creating Low-High variable
for index, row in df.iterrows():
    try:
        df["Low"][index] = ('+' + df["L_Strike"][index] + df["PnC"][index]) if df["L_Strike"][index] else df["Low"][index]
        df["High"][index] = ('-' + df["H_Strike"][index] + df["PnC"][index]) if df["H_Strike"][index] else df["High"][index]
        df["Low-High"][index] = df["Low"][index] + df["High"][index]
    except: pass


In [ ]:
df.tail(10)


In [ ]:
df = df.fillna('')
for index, row in df.iterrows():
    try:
        nstop = df["Stop"][index] if (df["Stop"][index] or df["Symbol"][index]!=df["Symbol"][index+1] or df["Expiration"][index]!=df["Expiration"][index+1]) else df["Stop"][index+1] if df["PnC"][index+1] == "C" else df["Entry"][index+1]
    except: pass


In [ ]:
df = df.fillna('')
df['Entry_Sign'] = df['PnC'].apply(lambda x: '<' if x == 'P' else '>')
df['Target_Sign'] = df['PnC'].apply(lambda x: '>' if x == 'P' else '<')
df['Stop_Sign'] = df['Entry_Sign']

#Filling all empty stop values from next row Empty values for same stock
for index, row in df.iterrows():
    try:
        df["Stop"][index] = df["Stop"][index] if (df["Stop"][index] or df["Symbol"][index]!=df["Symbol"][index+1] or df["Expiration"][index]!=df["Expiration"][index+1]) else df["Stop"][index+1] if df["PnC"][index+1] == "C" else df["Entry"][index+1]
    except: pass


In [ ]:
#Filling all empty stop/entry/target signs from previous row when it's the same
#Type/Symbol/Trend group (this is what already broadcasts a combo's shared sign
#across its legs, since every leg of one combo shares Type/Symbol/Trend)
for index, row in df.iterrows():
    try:
        df["Stop_Sign"][index] = df["Stop_Sign"][index-1] if (df["Type"][index] == df["Type"][index-1] and df["Symbol"][index] == df["Symbol"][index-1] and df["Trend"][index] == df["Trend"][index-1]) else df["Entry_Sign"][index]
    except: pass


In [ ]:
for index, row in df.iterrows():
    try:
        df["Entry_Sign"][index] = df["Entry_Sign"][index-1] if (df["Type"][index] == df["Type"][index-1] and df["Symbol"][index] == df["Symbol"][index-1] and df["Trend"][index] == df["Trend"][index-1]) else df["Entry_Sign"][index]
    except: pass


In [ ]:
for index, row in df.iterrows():
    try:
        df["Target_Sign"][index] = df["Target_Sign"][index-1] if (df["Type"][index] == df["Type"][index-1] and df["Symbol"][index] == df["Symbol"][index-1] and df["Trend"][index] == df["Trend"][index-1]) else df["Target_Sign"][index]
    except: pass


In [ ]:
df['Date'] = InputDate
df


In [ ]:
#Arranging columns as required in output file (quantity added at the end, per the new schema)
df = df[["Date","Type", "Trend", "Symbol", "Status", "Expiration", "PnC", "Low-High",
         "L_Strike" , "H_Strike", "Entry_Sign", "Entry", "Target_Sign", "Target",
         "Stop_Sign", "Stop", "quantity"]]

#Seperating the Investor and Trader Files
df_Investor = df[df["Trend"]=="Investor"]
df_Investor.reset_index(drop = True, inplace = True)
df_Trader = df[df["Trend"]=="Trader"]
df_Trader.reset_index(drop = True, inplace = True)


In [ ]:
df_Investor


In [ ]:
#Appending back both the Investor and Trader dataframes, one below the other
final_df = pd.concat([df_Investor, df_Trader], ignore_index=True, sort=False)
final_df = final_df.fillna('')
final_df


In [ ]:
from dotenv import load_dotenv
import logging
import numpy as np


In [ ]:
tf = file.replace('.pdf', '.csv')
print(tf)
final_df.to_csv(tf, sep=',', index=None)


In [ ]:
final_df = pd.read_csv(tf)
final_df


In [ ]:
final_df["Date"] = pd.to_datetime(final_df["Date"])
final_df["Expiration"] = pd.to_datetime(final_df["Expiration"])
low_high = final_df["Low-High"]  # keep pre-NaN copy: this column is NOT NULL in the DB
final_df = final_df.replace(r'^\s*$', np.nan, regex=True)
# "Low-High" is legitimately blank for strike-less rows (e.g. "Major US Market"
# index-summary lines with no option leg), but Trading.ETF_Options_v1 has it as NOT
# NULL — unlike Expiration/PnC/L_Strike/H_Strike/Entry/Target/Stop, which are nullable.
# Undo the blanket NaN-ification above for this one column so those rows insert as ""
# instead of tripping a "Column 'Low-High' cannot be null" IntegrityError.
final_df["Low-High"] = low_high
final_df = final_df.astype({'L_Strike':'float','H_Strike':'float','Target':'float','Entry':'float','Stop':'float','quantity':'int'})

final_df.info()


## Feature A — market commentary -> llmwiki (deferred)

`commentary_text` was captured from page 2's trailing prose when the PDF was split
earlier in this notebook. llmwiki isn't deployed yet (see `docs/TODOS.md`), so this
saves the commentary locally and only attempts the REST upload if `LLMWIKI_BASE_URL` is
configured — otherwise it logs and skips, exactly per the retry/skip design in
`docs/Design-Plan.md`.


In [ ]:
LLMWIKI_BASE_URL = os.environ.get("LLMWIKI_BASE_URL")
LLMWIKI_API_TOKEN = os.environ.get("LLMWIKI_API_TOKEN")
MAX_LLMWIKI_RETRY = int(os.environ.get("MAX_LLMWIKI_RETRY", "5"))

def upload_commentary_to_llmwiki(text: str, title: str) -> bool:
    """POST the market-commentary text to llmwiki's /upload endpoint as text/plain.

    Retries up to MAX_LLMWIKI_RETRY times on failure; on total failure, logs and gives
    up (returns False) rather than raising, so a llmwiki outage never aborts the
    notebook run (per docs/Design-Plan.md's retry/skip design).
    """
    url = f"{LLMWIKI_BASE_URL.rstrip('/')}/upload"
    headers = {"Authorization": f"Bearer {LLMWIKI_API_TOKEN}"}
    data = {"title": title}
    for attempt in range(1, MAX_LLMWIKI_RETRY + 1):
        try:
            files = {"file": (f"{title}.txt", text, "text/plain")}
            resp = requests.post(url, headers=headers, files=files, data=data, timeout=30)
            resp.raise_for_status()
            logging.info(f"llmwiki ingest succeeded on attempt {attempt}: {resp.text}")
            return True
        except Exception:
            logging.error(f"llmwiki upload attempt {attempt}/{MAX_LLMWIKI_RETRY} failed", exc_info=True)
    logging.error(f"llmwiki upload failed after {MAX_LLMWIKI_RETRY} attempts; skipping.")
    return False

commentary_title = f"LCR Playbook Market Commentary {InputDate}"
commentary_path = file.replace('.pdf', '_commentary.txt')
if commentary_text:
    with open(commentary_path, 'w') as f:
        f.write(commentary_text)
    print(f"Saved market commentary to {commentary_path}")
    if LLMWIKI_BASE_URL:
        upload_commentary_to_llmwiki(commentary_text, commentary_title)
    else:
        logging.info("LLMWIKI_BASE_URL not set; skipping llmwiki upload (Feature A not deployed yet).")
else:
    logging.warning("No market commentary text captured from page 2.")


## Load to MySQL — with a CSV-only escape hatch

`SKIP_DB_UPLOAD` defaults to `True`: the live `Trading.ETF_Options` table doesn't yet
have the new `quantity` column or the composite primary key described in
`docs/Design-Plan.md`/`docs/TODOS.md`, so an upload would either fail (unknown column)
or silently duplicate rows (no upsert yet). Flip it to `False` only after that migration
has landed.


In [ ]:
# Default True: the live table doesn't have `quantity` / the new composite PK yet
# (see docs/Design-Plan.md, docs/TODOS.md) — set to False only after that migration lands.
SKIP_DB_UPLOAD = False

if SKIP_DB_UPLOAD:
    print(f"SKIP_DB_UPLOAD=True — wrote CSV to {tf}; not uploading to database.")
else:
    load_dotenv(".env")  # Check path for env variables
    logging.getLogger().setLevel(logging.DEBUG)
    opt_tbl = "ETF_Options_v1"
    trd_DB = "Trading"
    DU.StoreEOD(final_df, trd_DB, opt_tbl)
    print(f"Uploaded {len(final_df)} rows to {trd_DB}.{opt_tbl}")
